In [2]:
import cv2
import os
import numpy as np
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization, Activation

In [9]:
import os
import cv2
import numpy as np

def UTK_read(path_folder):
    image_list = []
    gender_list = []
    age_list = []

    for file in os.listdir(path_folder):
        try:
            parts = file.split('_')
            if len(parts) < 2:
                continue
            
            gender = int(parts[1])
            age = int(parts[0])

            path_image = os.path.join(path_folder, file)
            image = cv2.imread(path_image)
            
            # Bỏ qua ảnh lỗi hoặc không mở được
            if image is None:
                continue

            image = cv2.resize(image, (64, 64))
            image = image / 255.0

            image_list.append(image)
            gender_list.append(gender)
            age_list.append(age)

        except Exception as e:
            # In ra thông báo lỗi (nếu cần debug), rồi bỏ qua ảnh
            continue

    X = np.array(image_list)
    y_gender = np.array(gender_list)
    y_age = np.array(age_list)
    return X, y_gender, y_age


In [10]:
utk_folder = '/kaggle/input/genderage-recognition/UTKFace'
X, y_gender, y_age = UTK_read(utk_folder)

In [12]:
from sklearn.model_selection import train_test_split

X_train, X_temp, y_age_train, y_age_temp, y_gender_train, y_gender_temp = train_test_split(
    X, y_age, y_gender, test_size=0.30, random_state=42
)

X_val, X_test, y_age_val, y_age_test, y_gender_val, y_gender_test = train_test_split(
    X_temp, y_age_temp, y_gender_temp, test_size=0.50, random_state=42
)

datagen = ImageDataGenerator(
    rotation_range=15,
    zoom_range=0.1,
    horizontal_flip=True,
    width_shift_range=0.1,     
    height_shift_range=0.1, 
    brightness_range=[0.8, 1.2]
)
datagen.fit(X_train)

In [15]:
input_shape = (64, 64, 3)
inputs = Input(shape=input_shape)


x = Conv2D(32, (3, 3), padding='same')(inputs)
x = Activation('relu')(x)
x = BatchNormalization()(x)
x = MaxPooling2D()(x)


x = Conv2D(64, (3, 3), padding='same')(x)
x = Activation('relu')(x)
x = BatchNormalization()(x)
x = MaxPooling2D()(x)

x = Conv2D(128, (3, 3), padding='same')(x)
x = Activation('relu')(x)
x = BatchNormalization()(x)
x = MaxPooling2D()(x)


x = Conv2D(256, (3, 3), padding='same')(x)
x = Activation('relu')(x)
x = BatchNormalization()(x)
x = MaxPooling2D()(x)

x = Conv2D(512, (3, 3), padding='same')(x)
x = Activation('relu')(x)
x = BatchNormalization()(x)
x = MaxPooling2D()(x)


x = Flatten()(x)
x = Dropout(0.1)(x)

output_gender = Dense(1, activation='sigmoid', name='gender')(x)
output_age = Dense(1, activation='linear', name='age')(x)

model = Model(inputs=inputs, outputs=[output_gender, output_age])
model.compile(optimizer='adam',
              loss={'gender': 'binary_crossentropy', 'age': 'mse'},
              metrics={'gender': 'accuracy', 'age': 'mae'})

model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)              ┃ Output Shape           ┃        Param # ┃ Connected to           ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1             │ (None, 64, 64, 3)      │              0 │ -                      │
│ (InputLayer)              │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv2d_5 (Conv2D)         │ (None, 64, 64, 32)     │            896 │ input_layer_1[0][0]    │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ activation_5 (Activation) │ (None, 64, 64, 32)     │              0 │ conv2d_5[0][0]         │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ batch_normalization_5     │ (None, 64, 64, 32)     │            128 │ activation_5[0][0]     │
│ (BatchNormalization)      │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ max_pooling2d_5           │ (None, 32, 32, 32)     │              0 │ batch_normalization_5… │
│ (MaxPooling2D)            │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv2d_6 (Conv2D)         │ (None, 32, 32, 64)     │         18,496 │ max_pooling2d_5[0][0]  │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ activation_6 (Activation) │ (None, 32, 32, 64)     │              0 │ conv2d_6[0][0]         │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ batch_normalization_6     │ (None, 32, 32, 64)     │            256 │ activation_6[0][0]     │
│ (BatchNormalization)      │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ max_pooling2d_6           │ (None, 16, 16, 64)     │              0 │ batch_normalization_6… │
│ (MaxPooling2D)            │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv2d_7 (Conv2D)         │ (None, 16, 16, 128)    │         73,856 │ max_pooling2d_6[0][0]  │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ activation_7 (Activation) │ (None, 16, 16, 128)    │              0 │ conv2d_7[0][0]         │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ batch_normalization_7     │ (None, 16, 16, 128)    │            512 │ activation_7[0][0]     │
│ (BatchNormalization)      │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ max_pooling2d_7           │ (None, 8, 8, 128)      │              0 │ batch_normalization_7… │
│ (MaxPooling2D)            │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv2d_8 (Conv2D)         │ (None, 8, 8, 256)      │        295,168 │ max_pooling2d_7[0][0]  │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ activation_8 (Activation) │ (None, 8, 8, 256)      │              0 │ conv2d_8[0][0]         │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ batch_normalization_8     │ (None, 8, 8, 256)      │          1,024 │ activation_8[0][0]     │
│ (BatchNormalization) 

 Total params: 1,576,642 (6.01 MB)

 Trainable params: 1,574,658 (6.01 MB)

 Non-trainable params: 1,984 (7.75 KB)

In [17]:
history = model.fit(
    X_train,
    {'gender': y_gender_train, 'age': y_age_train},
    validation_data=(X_val, {'gender': y_gender_val, 'age': y_age_val}),
    epochs=40,
    batch_size=32
)


Epoch 1/40
545/545 ━━━━━━━━━━━━━━━━━━━━ 6s 12ms/step - age_loss: 86.6689 - age_mae: 6.9819 - gender_accuracy: 0.7830 - gender_loss: 0.4723 - loss: 87.1412 - val_age_loss: 150.0267 - val_age_mae: 9.4747 - val_gender_accuracy: 0.7709 - val_gender_loss: 0.4706 - val_loss: 150.5927
Epoch 2/40
545/545 ━━━━━━━━━━━━━━━━━━━━ 6s 11ms/step - age_loss: 69.1170 - age_mae: 6.2287 - gender_accuracy: 0.7921 - gender_loss: 0.4561 - loss: 69.5731 - val_age_loss: 95.2612 - val_age_mae: 7.1600 - val_gender_accuracy: 0.8164 - val_gender_loss: 0.4040 - val_loss: 95.6717
Epoch 3/40
545/545 ━━━━━━━━━━━━━━━━━━━━ 6s 11ms/step - age_loss: 53.2666 - age_mae: 5.5361 - gender_accuracy: 0.8175 - gender_loss: 0.4056 - loss: 53.6722 - val_age_loss: 90.4926 - val_age_mae: 6.8042 - val_gender_accuracy: 0.8322 - val_gender_loss: 0.3649 - val_loss: 90.8519
Epoch 4/40
545/545 ━━━━━━━━━━━━━━━━━━━━ 6s 11ms/step - age_loss: 42.7195 - age_mae: 4.9803 - gender_accuracy: 0.8324 - gender_loss: 0.3720 - loss: 43.0915 - val_age_lo

In [18]:
test_results = model.evaluate(
    X_test,
    {'gender': y_gender_test, 'age': y_age_test},
    batch_size=32,
    verbose=1
)


117/117 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - age_loss: 71.8652 - age_mae: 6.0557 - gender_accuracy: 0.8817 - gender_loss: 0.4340 - loss: 72.2992


In [20]:
model.save('age_gender.keras')

In [21]:
from tensorflow.keras.models import load_model
model = load_model("age_gender.keras")

In [26]:
def dudoan(model, path_anh):
    # Đọc ảnh
    img = cv2.imread(path_anh)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (64, 64))
    

    img = img / 255.0
    img = np.expand_dims(img, axis=0) 
    gender_pred, age_pred = model.predict(img)

    gender_label = "Nữ" if gender_pred[0][0] >= 0.5 else "Nam"
    age = int(age_pred[0][0])

    print(f"Giới tính: {gender_label}")
    print(f"Tuổi dự đoán: {age}")

dudoan(model, "/kaggle/input/anhhhp/Screenshot_3.jpg")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
Giới tính: Nam
Tuổi dự đoán: 29
